# 123.3x Steady-state drawdown of the surface water level

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import timflow.steady as tfs
import timflow.transient as tft

from bruggeman.flow1d import bruggeman_123_32, bruggeman_123_33

In [ ]:
bruggeman_123_32

In [ ]:
help(bruggeman_123_32)

In [ ]:
t = 1  # time, d
h = -2  # change in head, m
k = 20.0  # hydraulic conductivity, m/d
D = 50.0  # thickness of aquifer, m
c = 1000  # leakage factor, d
eta = 10

Compare to the solution computed with timflow.

In [ ]:
ml = tft.ModelXsection(naq=1, tmin=1e-8, tmax=1e2)
aq = tft.XsectionMaq(
    ml,
    x1=-np.inf,
    x2=np.inf,
    kaq=k,
    c=[c],
    Saq=[1 / (eta * c * D)],
    z=[1, 0, -D],
    topboundary="semi",
)
ls = tft.River1D(ml, xls=0.0, tsandh=[(0, h)])
ml.solve(silent=True)

In [ ]:
x = np.linspace(0.0001, 800, 101)
t = np.array([1, 2, 5, 10, 30, 60, 120, 360, 720, 1440])  # minutes

# timflow solution every 10th x-value
h_ml = ml.headalongline(x[::10], 0.0, t / 60 / 24)

plt.figure(figsize=(10, 3))
for i, ti in enumerate(t):
    t_days = ti / 60 / 24  # convert to hours
    ha = bruggeman_123_32(x, t_days, h, k, D, c, eta)
    (ph,) = plt.plot(x, ha, label=f"t={ti:.0f} min.")
    plt.plot(x[::10], h_ml[0, i], marker="x", ls="none", c=ph.get_color())
plt.plot([], [], marker="x", ls="none", color="k", label="timflow")
plt.legend(loc=(0, 1), frameon=False, ncol=6, fontsize="small")
plt.xlabel("x [m]")
plt.grid()
plt.ylabel("drawdown [m]");

In [ ]:
x = np.array([2**i for i in range(4, 11)])
t = np.logspace(-8, 0, 101)

h_ml = ml.headalongline(x, 0.0, t[::10])

plt.figure(figsize=(10, 3))
for i, xi in enumerate(x):
    ha = bruggeman_123_32(xi, t, h, k, D, c, eta)
    (ph,) = plt.plot(t, ha, label=f"x={xi} m")
    plt.plot(t[::10], h_ml[0, :, i], marker="x", ls="none", color=ph.get_color())
plt.plot([], [], marker="x", ls="none", color="k", label="timflow")
plt.legend(loc=(0, 1), frameon=False, ncol=8, fontsize="small")
plt.xlabel("Time [d]")
plt.grid()
plt.ylabel("Drawdown [m]")
plt.xlim(1e-6, 1e0)
plt.ylim(h * 1.1, h * -0.1)
plt.xscale("log");

In [ ]:
bruggeman_123_33

In [ ]:
help(bruggeman_123_33)

In [ ]:
t = 1  # time, d
h = -2  # change in head, m
k = 20.0  # hydraulic conductivity, m/d
D = 50.0  # thickness of aquifer, m

Compare to the solution computed with timflow.

In [ ]:
def timflow_steady(x, c):
    ml = tfs.ModelXsection(naq=1)
    tfs.XsectionMaq(
        ml,
        x1=-np.inf,
        x2=np.inf,
        kaq=[k],
        c=[c],
        z=[1, 0, -D],
        topboundary="semi",
        hstar=0.0,
    )
    tfs.River1D(ml, xls=0.0, hls=h)
    ml.solve(silent=True)
    return ml.headalongline(x, 0.0)

In [ ]:
x = np.linspace(0, 2000, 100)
c = np.linspace(100, 500, 5)  # resitance, d
plt.figure(figsize=(10, 3))
for ci in c:
    ha = bruggeman_123_33(x, h, k, D, ci)
    (ph,) = plt.plot(x, ha, label=f"c={ci:.0f} days")
    h_ml = timflow_steady(x[::10], ci)
    plt.plot(x[::10], h_ml[0, :], marker="x", ls="none", c=ph.get_color())
plt.plot([], [], marker="x", ls="none", color="k", label="timflow")
plt.legend(loc=(0, 1), frameon=False, ncol=8, fontsize="small")
plt.grid()
plt.ylabel("Drawdown [m]")
plt.xlabel("Distance [m]");